# Dhanova synthetic-data EDA

This notebook inspects generated data, hard negatives, device sharing, and fraud-ring activity. The data is synthetic and is not evidence of real-world accuracy.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data = {name: pd.read_parquet(ROOT / 'data' / f'{name}.parquet') for name in ['accounts', 'devices', 'account_devices', 'transactions', 'labels']}
for name, frame in data.items():
    print(name, frame.shape)

In [ ]:
labels = data['labels']
print('Fraud prevalence:', labels['is_mule'].mean())
labels['ring_archetype'].value_counts(dropna=False)

In [ ]:
tx = data['transactions'].copy()
tx['is_mule_sender'] = tx['sender_account_id'].isin(set(labels.loc[labels.is_mule, 'account_id']))
tx['is_mule_receiver'] = tx['receiver_account_id'].isin(set(labels.loc[labels.is_mule, 'account_id']))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tx['amount'].clip(upper=50000).hist(ax=axes[0], bins=50)
axes[0].set_title('Transaction amounts, capped at ₹50,000')
tx.groupby('is_mule_sender')['amount'].median().plot.bar(ax=axes[1], title='Median amount by mule sender')
plt.tight_layout()

In [ ]:
device_counts = data['account_devices'].groupby('device_id')['account_id'].nunique()
device_counts.describe(), device_counts[device_counts > 2].sort_values(ascending=False).head(20)

In [ ]:
fraud_ids = set(labels.loc[labels.is_mule, 'account_id'])
fraud_tx = tx[tx.sender_account_id.isin(fraud_ids) | tx.receiver_account_id.isin(fraud_ids)]
fraud_tx['amount'].between(9000, 9999).mean(), fraud_tx.groupby(fraud_tx.timestamp.dt.floor('D')).size().sort_values(ascending=False).head(10)